# Hybrid search: BM25 and vector: the real-library code of the lessons

Booster LEAP, Advanced RAG Engineering. The cells below are the library versions of the examples in the lessons (the lessons first build the same mechanism in plain Python in your browser). Each cell was run once and its output is saved below it.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use it:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again.

**What needs what:** the first run downloads the small embedding model all-MiniLM-L6-v2 (about 90 MB; it needs internet but no GPU), and chromadb is installed for lesson 5 (about 100 MB of packages). No secret or key is used in this notebook. The install line pins the versions that were used to produce the saved outputs; Colab may resolve the dependencies of those packages differently.

In [ ]:
%pip -q install rank-bm25==0.2.2 sentence-transformers==6.3.0 chromadb==1.5.9

## When a keyword search wins

The three nearest passages with cosine similarity for a code, a number and a paraphrase (real model, run in VS Code).

In [ ]:
from sentence_transformers import SentenceTransformer

docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}

queries = {"q01": "ERR-5103", "q06": "10,000 tickets", "q10": "can I paste client information into ChatGPT"}

model = SentenceTransformer("all-MiniLM-L6-v2")
ids = list(docs)
pv = model.encode([docs[i] for i in ids], normalize_embeddings=True)
for qid, text in queries.items():
    qv = model.encode([text], normalize_embeddings=True)[0]
    sims = pv @ qv                      # unit-length vectors: the dot product is the cosine
    order = sorted(range(len(ids)), key=lambda j: (-float(sims[j]), ids[j]))
    print(qid, text)
    for j in order[:3]:
        print(f"  {sims[j]:.3f}  {ids[j]}")

q01 ERR-5103
  0.341  kb-err4013
  0.323  kb-err4021
  0.316  kb-err5032
q06 10,000 tickets
  0.428  kb-merge
  0.407  kb-slow
  0.385  faq-trash
q10 can I paste client information into ChatGPT
  0.395  sec-data
  0.125  sec-passwords
  0.120  faq-region


## BM25: how a keyword score is built

rank_bm25 on the same passages and questions: the idf, the ranks, and the effect of a repeated question word (run in VS Code).

In [ ]:
import math
from rank_bm25 import BM25Okapi

import re

STOP = set("a an the of to in on at for from by with and or is are was be do does did can i my me we you it its this that what how who which when where if not no as into than then there their they them your our have has had will may must".split())

def tokenize(text, drop_stop=True):
    """lower case words and numbers (a code such as ERR-4012 becomes the two words err and 4012); common function words are dropped unless drop_stop is False"""
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [w for w in words if not (drop_stop and w in STOP)]

docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}

queries = [
    ("q01", "ERR-5103", "kb-err5103"), ("q02", "ERR-5031", "kb-err5031"),
    ("q03", "ERR-4012", "kb-err4012"), ("q04", "Sev 2 response time", "run-sev"),
    ("q05", "Frankfurt", "faq-region"), ("q06", "10,000 tickets", "kb-slow"),
    ("q07", "Tier 2 dinner limit", "exp-meals"), ("q08", "31 March", "leave-carry"),
    ("q09", "how much time off do I get each year", "leave-annual"), ("q10", "can I paste client information into ChatGPT", "sec-data"),
    ("q11", "I am expecting a baby, how long can I stay home", "leave-parental"), ("q12", "can I fly in a fancier seat", "exp-travel"),
    ("q13", "messages from customers do not show up", "kb-mail"), ("q14", "the app is very slow", "kb-slow"),
    ("q15", "the website keeps showing the login again and again", "kb-login-loop"), ("q16", "what if my work computer is taken", "sec-devices"),
]

ids = list(docs)
bm = BM25Okapi([tokenize(docs[i]) for i in ids], k1=1.5, b=0.75)
print("rank_bm25: epsilon", bm.epsilon, "| average idf", round(bm.average_idf, 3))
print("word      n   lesson idf  library idf  ln((N-n+0.5)/(n+0.5))")
for word in ["5103", "customer", "err"]:
    n = sum(word in tokenize(t) for t in docs.values())
    plain = math.log((bm.corpus_size - n + 0.5) / (n + 0.5))
    mine = math.log(1 + (bm.corpus_size - n + 0.5) / (n + 0.5))
    print(f"{word:9} {n}   {mine:10.3f}  {bm.idf[word]:11.3f}  {plain:8.3f}")

ranks = []
for qid, text, right in queries:
    scores = bm.get_scores(tokenize(text))
    order = [ids[j] for j in sorted(range(len(ids)), key=lambda j: (-scores[j], ids[j])) if scores[j] > 0]
    ranks.append(order.index(right) + 1 if right in order else None)
print("rank of the right passage for q01 to q16:", ranks)

words = tokenize("the website keeps showing the login again and again")
for label, w in [("question words as typed", words), ("each word once", sorted(set(words)))]:
    scores = bm.get_scores(w)
    order = sorted(range(len(ids)), key=lambda j: (-scores[j], ids[j]))[:3]
    print(f"{label:24}", [(ids[j], round(float(scores[j]), 2)) for j in order])

tiny = BM25Okapi([["the", "cat"], ["the", "dog"], ["the", "bird"], ["a", "fish"]])
print("tiny collection: ln((4 - 3 + 0.5) / (3 + 0.5)) =", round(math.log((4 - 3 + 0.5) / (3 + 0.5)), 3), "| library idf of 'the' =", round(tiny.idf["the"], 3))

rank_bm25: epsilon 0.25 | average idf 2.744
word      n   lesson idf  library idf  ln((N-n+0.5)/(n+0.5))
5103      1        2.962        2.909     2.909
customer  4        1.863        1.695     1.695
err       6        1.495        1.242     1.242
rank of the right passage for q01 to q16: [1, 1, 1, 1, 1, 1, 1, 1, 1, None, None, None, None, None, 1, None]
question words as typed  [('kb-login-loop', 7.78), ('kb-err5103', 4.84), ('kb-merge', 2.79)]
each word once           [('kb-login-loop', 5.18), ('kb-merge', 2.79), ('kb-err5103', 2.42)]
tiny collection: ln((4 - 3 + 0.5) / (3 + 0.5)) = -0.847 | library idf of 'the' = 0.141


## What a store offers for hybrid search

Inspect chromadb 1.5.9: its query parameters, the rank-fusion objects and what the local client does with them.

In [ ]:
import inspect
import chromadb
from chromadb.execution.expression import Knn, Rrf, Search

print("chromadb", chromadb.__version__)
col = chromadb.EphemeralClient().create_collection("probe-hybrid")
col.add(ids=["a", "b"], embeddings=[[1.0, 0.0], [0.0, 1.0]], documents=["error ERR-5103 on import", "customer emails in spam"])
print("query() parameters:", [p for p in inspect.signature(col.query).parameters if p.startswith(("query_", "where"))])
print("has search():", hasattr(col, "search"), "| Rrf parameters:", list(inspect.signature(Rrf).parameters))
print("keyword filter:", col.query(query_embeddings=[[1.0, 0.0]], n_results=2, where_document={"$contains": "spam"})["ids"])
try:
    col.search(Search().rank(Rrf([Knn(query=[1.0, 0.0], return_rank=True), Knn(query=[0.0, 1.0], return_rank=True)])).limit(2))
except Exception as e:
    print("search() on the local client:", type(e).__name__, "-", e)

chromadb 1.5.9
query() parameters: ['query_embeddings', 'query_texts', 'query_images', 'query_uris', 'where', 'where_document']
has search(): True | Rrf parameters: ['ranks', 'k', 'weights', 'normalize']
keyword filter: [['b']]
search() on the local client: NotImplementedError - Search is not implemented for Local Chroma


## Tuning hybrid search by measurement

The real model on the 28 passages: vector size, dense recall@3 on the 16 tuning queries, and the time of one query encoding and one cosine step (times differ from computer to computer).

In [ ]:
docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
queries = {
    "q01": "ERR-5103",
    "q02": "ERR-5031",
    "q03": "ERR-4012",
    "q04": "Sev 2 response time",
    "q05": "Frankfurt",
    "q06": "10,000 tickets",
    "q07": "Tier 2 dinner limit",
    "q08": "31 March",
    "q09": "how much time off do I get each year",
    "q10": "can I paste client information into ChatGPT",
    "q11": "I am expecting a baby, how long can I stay home",
    "q12": "can I fly in a fancier seat",
    "q13": "messages from customers do not show up",
    "q14": "the app is very slow",
    "q15": "the website keeps showing the login again and again",
    "q16": "what if my work computer is taken",
}
gold = {"q01": "kb-err5103", "q02": "kb-err5031", "q03": "kb-err4012", "q04": "run-sev", "q05": "faq-region", "q06": "kb-slow", "q07": "exp-meals", "q08": "leave-carry", "q09": "leave-annual", "q10": "sec-data", "q11": "leave-parental", "q12": "exp-travel", "q13": "kb-mail", "q14": "kb-slow", "q15": "kb-login-loop", "q16": "sec-devices"}

import time
from sentence_transformers import SentenceTransformer

ids = list(docs)
model = SentenceTransformer("all-MiniLM-L6-v2")
vectors = model.encode([docs[i] for i in ids], normalize_embeddings=True)
print("vectors:", vectors.shape, vectors.dtype, vectors.nbytes, "bytes")

hits = 0
for q, text in queries.items():
    scores = vectors @ model.encode([text], normalize_embeddings=True)[0]
    top3 = [ids[j] for j in sorted(range(len(ids)), key=lambda j: (-float(scores[j]), ids[j]))[:3]]
    hits += gold[q] in top3
print("dense side alone: right passage in the top 3 for", hits, "of", len(queries), "queries")

def median_ms(fn, runs=20):
    times = []
    for _ in range(runs):
        t = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t) * 1000)
    return sorted(times)[runs // 2]

q_vec = model.encode(["can I fly in a fancier seat"], normalize_embeddings=True)[0]
print("encode one query:        about", round(median_ms(lambda: model.encode(["can I fly in a fancier seat"], normalize_embeddings=True)), 2), "ms")
print("cosine against 28 vectors: about", round(median_ms(lambda: vectors @ q_vec), 4), "ms")

vectors: (28, 384) float32 43008 bytes
dense side alone: right passage in the top 3 for 15 of 16 queries
encode one query:        about 12.94 ms
cosine against 28 vectors: about 0.0028 ms


## Notebook: a hybrid search program from end to end

The script that made the saved dense scores, with its real output.

In [ ]:
docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
queries = {
    "q01": ("ERR-5103", "kb-err5103"),
    "q02": ("ERR-5031", "kb-err5031"),
    "q03": ("ERR-4012", "kb-err4012"),
    "q04": ("Sev 2 response time", "run-sev"),
    "q05": ("Frankfurt", "faq-region"),
    "q06": ("10,000 tickets", "kb-slow"),
    "q07": ("Tier 2 dinner limit", "exp-meals"),
    "q08": ("31 March", "leave-carry"),
    "q09": ("how much time off do I get each year", "leave-annual"),
    "q10": ("can I paste client information into ChatGPT", "sec-data"),
    "q11": ("I am expecting a baby, how long can I stay home", "leave-parental"),
    "q12": ("can I fly in a fancier seat", "exp-travel"),
    "q13": ("messages from customers do not show up", "kb-mail"),
    "q14": ("the app is very slow", "kb-slow"),
    "q15": ("the website keeps showing the login again and again", "kb-login-loop"),
    "q16": ("what if my work computer is taken", "sec-devices"),
    "h01": ("ERR-4013", "kb-err4013"),
    "h02": ("99.9 uptime", "faq-uptime"),
    "h03": ("how long can I still recover deleted items", "faq-trash"),
    "h04": ("can the other parent take time off after the birth", "leave-parental"),
    "h05": ("ERR-5032 link", "kb-err5032"),
    "h06": ("who approves a first class flight", "exp-travel"),
    "h07": ("Starter seats", "faq-seats"),
    "h08": ("are my emails going to junk", "kb-mail"),
}

import hashlib
import json
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
ids = list(docs)
passage_vectors = model.encode([docs[i] for i in ids], normalize_embeddings=True)      # one vector per passage, length 1
print("passage vectors:", passage_vectors.shape)

saved = {}
for q, (text, right) in queries.items():
    query_vector = model.encode([text], normalize_embeddings=True)[0]
    cosines = passage_vectors @ query_vector                                         # cosine, because both are length 1
    order = sorted(range(len(ids)), key=lambda j: (-float(cosines[j]), ids[j]))
    saved[q] = [[ids[j], round(float(cosines[j]), 4)] for j in order]                 # the saved file: all passages, best first

text = json.dumps(saved, indent=0)
print("saved file: 24 queries x 28 passages, digest", hashlib.sha256(text.encode()).hexdigest()[:12])
for q, (_, right) in queries.items():
    print(q, "right passage at rank", [p for p, _ in saved[q]].index(right) + 1, "| best:", saved[q][0][0], saved[q][0][1])

passage vectors: (28, 384)
saved file: 24 queries x 28 passages, digest f6a2db2873ae
q01 right passage at rank 4 | best: kb-err4013 0.3409
q02 right passage at rank 3 | best: kb-err5032 0.4016
q03 right passage at rank 3 | best: kb-err4013 0.5731
q04 right passage at rank 1 | best: run-sev 0.757
q05 right passage at rank 1 | best: faq-region 0.4418
q06 right passage at rank 2 | best: kb-merge 0.4281
q07 right passage at rank 1 | best: exp-meals 0.6754
q08 right passage at rank 1 | best: leave-carry 0.3101
q09 right passage at rank 1 | best: leave-annual 0.4797
q10 right passage at rank 1 | best: sec-data 0.395
q11 right passage at rank 1 | best: leave-parental 0.3711
q12 right passage at rank 1 | best: exp-travel 0.2922
q13 right passage at rank 1 | best: kb-mail 0.5669
q14 right passage at rank 1 | best: kb-slow 0.3344
q15 right passage at rank 1 | best: kb-login-loop 0.5035
q16 right passage at rank 1 | best: sec-devices 0.4042
h01 right passage at rank 1 | best: kb-err4013 0.6429
h0

The system table with rank_bm25 and the real model, and a measured comparison of the two BM25 versions.

In [ ]:
docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
queries = {
    "q01": ("ERR-5103", "kb-err5103"),
    "q02": ("ERR-5031", "kb-err5031"),
    "q03": ("ERR-4012", "kb-err4012"),
    "q04": ("Sev 2 response time", "run-sev"),
    "q05": ("Frankfurt", "faq-region"),
    "q06": ("10,000 tickets", "kb-slow"),
    "q07": ("Tier 2 dinner limit", "exp-meals"),
    "q08": ("31 March", "leave-carry"),
    "q09": ("how much time off do I get each year", "leave-annual"),
    "q10": ("can I paste client information into ChatGPT", "sec-data"),
    "q11": ("I am expecting a baby, how long can I stay home", "leave-parental"),
    "q12": ("can I fly in a fancier seat", "exp-travel"),
    "q13": ("messages from customers do not show up", "kb-mail"),
    "q14": ("the app is very slow", "kb-slow"),
    "q15": ("the website keeps showing the login again and again", "kb-login-loop"),
    "q16": ("what if my work computer is taken", "sec-devices"),
    "h01": ("ERR-4013", "kb-err4013"),
    "h02": ("99.9 uptime", "faq-uptime"),
    "h03": ("how long can I still recover deleted items", "faq-trash"),
    "h04": ("can the other parent take time off after the birth", "leave-parental"),
    "h05": ("ERR-5032 link", "kb-err5032"),
    "h06": ("who approves a first class flight", "exp-travel"),
    "h07": ("Starter seats", "faq-seats"),
    "h08": ("are my emails going to junk", "kb-mail"),
}
import re

STOP = set("a an the of to in on at for from by with and or is are was be do does did can i my me we you it its this that what how who which when where if not no as into than then there their they them your our have has had will may must".split())

def tokenize(text, drop_stop=True):
    """lower case words and numbers (a code such as ERR-4012 becomes the two words err and 4012); common function words are dropped unless drop_stop is False"""
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [w for w in words if not (drop_stop and w in STOP)]
import math

def build_bm25(docs, k1=1.5, b=0.75):
    """docs: {id: text}. Returns an index with term counts, lengths and document frequencies."""
    toks = {pid: tokenize(t) for pid, t in docs.items()}
    n = len(toks)
    avg = sum(len(t) for t in toks.values()) / n
    df = {}
    for t in toks.values():
        for w in set(t):
            df[w] = df.get(w, 0) + 1
    return {"toks": toks, "n": n, "avg": avg, "df": df, "k1": k1, "b": b}

def idf(index, word):
    df = index["df"].get(word, 0)
    return math.log(1 + (index["n"] - df + 0.5) / (df + 0.5))

def bm25_scores(index, query):
    k1, b = index["k1"], index["b"]
    scores = {}
    for pid, toks in index["toks"].items():
        s = 0.0
        for w in set(tokenize(query)):
            tf = toks.count(w)
            if tf:
                s += idf(index, w) * tf * (k1 + 1) / (tf + k1 * (1 - b + b * len(toks) / index["avg"]))
        scores[pid] = s
    return scores

def bm25_rank(index, query):
    """passage ids best first; passages with score 0 are left out"""
    s = bm25_scores(index, query)
    return [pid for pid in sorted(s, key=lambda p: (-s[p], p)) if s[pid] > 0]
def rrf(rankings, k=60):
    """reciprocal rank fusion: each list gives 1 / (k + rank) to its passages (rank starts at 1); returns ids best first"""
    score = {}
    for ranking in rankings:
        for rank, pid in enumerate(ranking, 1):
            score[pid] = score.get(pid, 0.0) + 1.0 / (k + rank)
    return sorted(score, key=lambda p: (-score[p], p))
def min_max(scores):
    """scale a {id: score} dictionary to 0..1 (all zeros when every score is equal)"""
    lo, hi = min(scores.values()), max(scores.values())
    return {pid: (s - lo) / (hi - lo) if hi > lo else 0.0 for pid, s in scores.items()}

def weighted_fusion(sparse, dense, alpha):
    """sparse and dense: {id: score} for the same passages. alpha is the weight of the dense side (0 = sparse only, 1 = dense only).
    Returns passage ids best first (ties by id)."""
    s, d = min_max(sparse), min_max(dense)
    fused = {pid: alpha * d[pid] + (1 - alpha) * s[pid] for pid in s}
    return sorted(fused, key=lambda p: (-fused[p], p))
def evaluate(rankings, gold, k=3):
    """rankings: {query id: [passage ids best first]}, gold: {query id: the one right passage id}. Returns (recall@k, MRR)."""
    hit, rr = 0, 0.0
    for q, right in gold.items():
        ranked = rankings.get(q, [])
        if right in ranked[:k]:
            hit += 1
        if right in ranked:
            rr += 1.0 / (ranked.index(right) + 1)
    return hit / len(gold), rr / len(gold)

import math
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

ids = list(docs)
model = SentenceTransformer("all-MiniLM-L6-v2")
passage_vectors = model.encode([docs[i] for i in ids], normalize_embeddings=True)
own = build_bm25(docs)                                              # the lesson's BM25
library = BM25Okapi([tokenize(docs[i]) for i in ids])                # rank_bm25, same tokens, same k1 = 1.5 and b = 0.75

own_s, lib_s, dense_s = {}, {}, {}
for q, (text, right) in queries.items():
    own_s[q] = bm25_scores(own, text)
    lib_s[q] = dict(zip(ids, (float(x) for x in library.get_scores(tokenize(text)))))
    dense_s[q] = dict(zip(ids, (float(x) for x in passage_vectors @ model.encode([text], normalize_embeddings=True)[0])))

def keyword_order(scores):
    return [p for p in sorted(scores, key=lambda p: (-scores[p], p)) if scores[p] > 0]

gold = {q: right for q, (_, right) in queries.items()}
tuning = [q for q in queries if q.startswith("q")]
held_out = [q for q in queries if q.startswith("h")]

def table(name, rank):
    r1, m1 = evaluate({q: rank(q) for q in tuning}, {q: gold[q] for q in tuning}, 3)
    r2, m2 = evaluate({q: rank(q) for q in held_out}, {q: gold[q] for q in held_out}, 3)
    print(f"{name:26s} {r1:8.4f} {m1:7.4f} {r2:10.4f} {m2:7.4f}")

print("system                     tuning recall@3  MRR  held-out recall@3  MRR")
table("BM25 (rank_bm25) alone", lambda q: keyword_order(lib_s[q]))
table("dense alone", lambda q: sorted(dense_s[q], key=lambda p: (-dense_s[q][p], p)))
table("RRF, k = 60 (rank_bm25)", lambda q: rrf([keyword_order(lib_s[q]), sorted(dense_s[q], key=lambda p: (-dense_s[q][p], p))]))
table("weighted 0.7 (rank_bm25)", lambda q: weighted_fusion(lib_s[q], dense_s[q], 0.7))
table("weighted 0.7 (own BM25)", lambda q: weighted_fusion(own_s[q], dense_s[q], 0.7))

print()
print("own BM25 against rank_bm25 on the 24 queries")
differ = [q for q in queries if any(abs(own_s[q][p] - lib_s[q][p]) > 1e-9 for p in ids)]
print("queries where any score differs:", len(differ), "of 24")
worst = max(queries, key=lambda q: max(abs(own_s[q][p] - lib_s[q][p]) for p in ids))
print("largest score difference:", round(max(abs(own_s[worst][p] - lib_s[worst][p]) for p in ids), 3), "in", worst)
print("order of the matching passages differs in:", [q for q in queries if keyword_order(own_s[q]) != keyword_order(lib_s[q])])
print("top 3 differs in:", [q for q in queries if keyword_order(own_s[q])[:3] != keyword_order(lib_s[q])[:3]])
print("after weighted fusion at 0.7, the full order differs in:", [q for q in queries if weighted_fusion(own_s[q], dense_s[q], 0.7) != weighted_fusion(lib_s[q], dense_s[q], 0.7)])
print("after weighted fusion at 0.7, the top 3 differs in:", [q for q in queries if weighted_fusion(own_s[q], dense_s[q], 0.7)[:3] != weighted_fusion(lib_s[q], dense_s[q], 0.7)[:3]])
print("repeated query words:", {q: tokenize(t) for q, (t, _) in queries.items() if len(set(tokenize(t))) < len(tokenize(t))})
for q in ("q15",):
    print(q, "own:    ", [(p, round(own_s[q][p], 2)) for p in keyword_order(own_s[q])[:3]])
    print(q, "library:", [(p, round(lib_s[q][p], 2)) for p in keyword_order(lib_s[q])[:3]])
word = "frankfurt"
print("idf of", word, ": own", round(math.log(1 + (28 - 1 + 0.5) / (1 + 0.5)), 3), "| library", round(library.idf[word], 3))

system                     tuning recall@3  MRR  held-out recall@3  MRR
BM25 (rank_bm25) alone       0.6250  0.6250     0.8750  0.8750
dense alone                  0.9375  0.8385     1.0000  1.0000
RRF, k = 60 (rank_bm25)      0.9375  0.9152     0.8750  0.8906
weighted 0.7 (rank_bm25)     1.0000  1.0000     1.0000  1.0000
weighted 0.7 (own BM25)      1.0000  1.0000     1.0000  1.0000

own BM25 against rank_bm25 on the 24 queries
queries where any score differs: 20 of 24
largest score difference: 2.394 in q15
order of the matching passages differs in: ['q15']
top 3 differs in: ['q15']
after weighted fusion at 0.7, the full order differs in: ['q13', 'q15', 'h04']
after weighted fusion at 0.7, the top 3 differs in: []
repeated query words: {'q15': ['website', 'keeps', 'showing', 'login', 'again', 'again']}
q15 own:     [('kb-login-loop', 5.38), ('kb-merge', 2.9), ('kb-err5103', 2.51)]
q15 library: [('kb-login-loop', 7.78), ('kb-err5103', 4.84), ('kb-merge', 2.79)]
idf of frankfurt : own 2